# Transfer Learning with MobileNetV2

## الترتيب / Flow
1. استيراد المكتبات - Import libraries
2. قراءة البيانات - Load MNIST sample CSV
3. تجهيز الصور - Resize to 224x224 RGB + normalize
4. تقسيم البيانات - Train/Test split
5. بناء MobileNetV2 - Pretrained base + new head
6. تجميد Base - Freeze pretrained layers
7. تدريب الرأس - Train classifier head
8. التقييم - Test accuracy
9. عرض تنبؤات - Sample predictions

In [ ]:
# Step 1) استيراد المكتبات / Import libraries
# pip install tensorflow -q  # uncomment in Colab if needed
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input

In [ ]:
# Step 2) قراءة البيانات / Load dataset
dataset = pd.read_csv('mnist_sample.csv')
print('Shape:', dataset.shape)
dataset.head()

In [ ]:
# Step 3) تجهيز الصور / Prepare images for MobileNetV2 (224x224 RGB)
y = dataset['label'].values.astype(int)
pixel_cols = [c for c in dataset.columns if c.startswith('pixel_')]
X_gray = dataset[pixel_cols].values.reshape(-1, 28, 28, 1).astype('float32')

# Grayscale -> RGB and resize to 224x224
X_rgb = np.repeat(X_gray, 3, axis=-1)
X_resized = tf.image.resize(X_rgb, (224, 224)).numpy()
X_resized = preprocess_input(X_resized)
print('Prepared shape:', X_resized.shape)

In [ ]:
# Step 4) تقسيم البيانات / Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resized, y, test_size=0.2, random_state=0, stratify=y
)

In [ ]:
# Step 5) بناء MobileNetV2 / Build transfer learning model
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
outputs = Dense(10, activation='softmax')(x)
model = Model(inputs, outputs)
model.summary()

In [ ]:
# Step 6) تجميد Base / Freeze pretrained layers
base_model.trainable = False
print(f'Trainable layers: {sum([l.trainable for l in model.layers])}')

In [ ]:
# Step 7) تدريب الرأس / Train classifier head only
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=16,
    verbose=1
)

In [ ]:
# Step 8) التقييم / Evaluate
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss: {loss:.4f}')
print(f'Test accuracy: {accuracy:.2%}')

In [ ]:
# Step 9) عرض تنبؤات / Visualize predictions
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_test[i].astype('float32') * 0.5 + 0.5)  # rough de-normalize for display
    color = 'green' if y_pred[i] == y_test[i] else 'red'
    ax.set_title(f'True: {y_test[i]} | Pred: {y_pred[i]}', color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('Transfer Learning Predictions (MobileNetV2)')
plt.tight_layout()
plt.show()